# 03 -- AAII Sentiment Data Collection

## Source
American Association of Individual Investors (AAII) weekly sentiment survey, read from a locally downloaded Excel file (`AAII_sentiment_pre.xls`, `.xls` format parsed with the `xlrd` engine).

## Collection Method
The raw Excel file is read with the first four rows skipped (address and multi-line header rows). Fully empty rows are dropped, and column names are manually assigned based on the spreadsheet layout.

## Variables Collected
- `date` -- survey date
- `bullish` -- % bullish respondents
- `neutral` -- % neutral respondents
- `bearish` -- % bearish respondents
- `total` -- total (should sum to 100%)
- `bullish_8w_ma` -- 8-week moving average of bullish %
- `bull_bear_spread` -- bullish % minus bearish %
- `bullish_avg` -- historical average of bullish %
- `bullish_avg_plus_sd` -- historical average + 1 standard deviation
- `bullish_avg_minus_sd` -- historical average - 1 standard deviation
- `sp500_weekly_high` -- S&P 500 weekly high
- `sp500_weekly_low` -- S&P 500 weekly low
- `sp500_weekly_close` -- S&P 500 weekly close

## Parsing
- Dates are coerced to datetime; rows with unparseable dates are dropped.
- Percentage columns are string-cleaned (removing `%` symbols) and cast to numeric.

## Filtering
Data is filtered to 2004-01-01 through 2024-12-31.

## Output
`Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_post` (Parquet)

In [6]:
import pandas as pd
import numpy as np

# ── Paths ────────────────────────────────────────────────────────────────────
AAII_PATH = r"C:/Users/Henry/OneDrive/Documents/LSE/MSc/Dissertation/Project/Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_pre.xls"
OUT_PATH  = "../../Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_post"

# ── 1. Read the AAII Excel file ──────────────────────────────────────────────
aaii_raw = pd.read_excel(
    AAII_PATH,
    header=None,
    skiprows=4,       # skip address + multi-line header rows
    engine="xlrd",    # .xls format
)

aaii_raw = aaii_raw.dropna(how="all").reset_index(drop=True)

# Assign column names from screenshot (columns A–M)
col_names = [
    "date",
    "bullish",
    "neutral",
    "bearish",
    "total",
    "bullish_8w_ma",
    "bull_bear_spread",
    "bullish_avg",
    "bullish_avg_plus_sd",
    "bullish_avg_minus_sd",
    "sp500_weekly_high",
    "sp500_weekly_low",
    "sp500_weekly_close",
]

# Handle any extra columns beyond the 13 visible
if len(aaii_raw.columns) > len(col_names):
    col_names += [f"extra_{i}" for i in range(len(aaii_raw.columns) - len(col_names))]
aaii_raw.columns = col_names[:len(aaii_raw.columns)]

aaii = aaii_raw.copy()

# ── 2. Parse dates ───────────────────────────────────────────────────────────
aaii["date"] = pd.to_datetime(aaii["date"], errors="coerce")
aaii = aaii.dropna(subset=["date"])

# ── 3. Parse percentage columns ──────────────────────────────────────────────
pct_cols = [
    "bullish", "neutral", "bearish", "total",
    "bullish_8w_ma", "bull_bear_spread",
    "bullish_avg", "bullish_avg_plus_sd", "bullish_avg_minus_sd",
]
for col in pct_cols:
    if col in aaii.columns:
        if aaii[col].dtype == object:
            aaii[col] = (
                aaii[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .apply(pd.to_numeric, errors="coerce")
            )

# ── 4. Filter to 2004-01-01 → 2024-12-31 ────────────────────────────────────
aaii = aaii[
    (aaii["date"] >= "2004-01-01") &
    (aaii["date"] <= "2024-12-31")
].reset_index(drop=True)

# ── 5. Quick summary ─────────────────────────────────────────────────────────
print(f"AAII Sentiment: {aaii.shape[0]} rows, {aaii.shape[1]} columns")
print(f"Date range: {aaii['date'].min().date()} → {aaii['date'].max().date()}")
print(f"\nColumns: {list(aaii.columns)}")
print(f"\nNulls:\n{aaii.isna().sum()}")
print(f"\n{aaii.head()}")

# ── 6. Save ──────────────────────────────────────────────────────────────────
aaii.to_parquet(OUT_PATH, index=False, engine="pyarrow")
print(f"\nSaved {OUT_PATH}: {aaii.shape}")

AAII Sentiment: 1095 rows, 14 columns
Date range: 2004-01-01 → 2024-12-26

Columns: ['date', 'bullish', 'neutral', 'bearish', 'total', 'bullish_8w_ma', 'bull_bear_spread', 'bullish_avg', 'bullish_avg_plus_sd', 'bullish_avg_minus_sd', 'sp500_weekly_high', 'sp500_weekly_low', 'sp500_weekly_close', 'extra_0']

Nulls:
date                       0
bullish                    0
neutral                    0
bearish                    0
total                      0
bullish_8w_ma              0
bull_bear_spread           0
bullish_avg                0
bullish_avg_plus_sd        0
bullish_avg_minus_sd       0
sp500_weekly_high          0
sp500_weekly_low           0
sp500_weekly_close         0
extra_0                 1095
dtype: int64

        date  bullish  neutral  bearish   total  bullish_8w_ma  \
0 2004-01-01   0.6241   0.2411   0.1348  1.0000       0.599625   
1 2004-01-08   0.6716   0.1493   0.1791  1.0000       0.616912   
2 2004-01-15   0.6629   0.2360   0.1011  1.0000       0.633237   
